### Fe構造のクラスタリング

炭素構造の場合は大まかにはsp,sp2,sp3結合を取ることがわかっているため、
炭素間結合の距離がほぼ決まっており、三次元構造から炭素構造を理解することが容易にできる。
金属はその結合が明確で無いため、目視による構造の特定が困難です。
金属の場合に構造から記述子を作成して構造のクラスタリングが可能かどうかが調べます。

このscriptではデータを全てDataFrameに入れています。

**データ取得からデータ解析**

In [ ]:
from typing import List, Tuple
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 10)


In [ ]:
import os
os.makedirs("image_executed", exist_ok=True)

In [ ]:
ACTCION = ["km", "gmm"]# kmeanとGMMを行う。

In [ ]:
def get_data(descriptor_set):
    """データ取得

    Args:
        descriptor_set (str)): 説明変数名のリスト

    Returns:
        pd.DataFrame: 観測データ。
        pd.DataFrame: 新規データ。
        [str]: 説明変数名。
        int: cluster数。
        int|None: 圧縮後次元 or None(次元圧縮しない)
    """
    # データ取得（読み込み）
    df_obs = pd.read_csv("../data_calculated/Fe2_descriptor.csv")
    df_new = pd.read_csv("../data_calculated/Fe2_descriptor_newdata.csv")

    nclusters = 3

    ndim_pre = None

    if descriptor_set=="a_rp":
        descriptor_names = ['a0.70_rp2.40', 'a0.70_rp3.00', 'a0.70_rp3.60', 'a0.70_rp4.20',
                        'a0.70_rp4.80', 'a0.70_rp5.40']
    elif descriptor_set=="hist":
        descriptor_names = ['hist1', 'hist2', 'hist3', 'hist4',
        'hist5', 'hist6', 'hist7', 'hist8', 'hist9', 'hist10', 'hist11',
        'hist12', 'hist13', 'hist14', 'hist15', 'hist16', 'hist17', 'hist18',
        'hist19', 'hist20',]
        
    print("descriptor_names", descriptor_names)

    return df_obs, df_new, descriptor_names, nclusters, ndim_pre

DESCRIPTOR_SET = "a_rp" # a_rp or hist
g_df_obs, g_df_new, g_descriptor_names, g_nclusters, g_ndim_pre = get_data(DESCRIPTOR_SET)

In [ ]:
def add_scaledX(df : pd.DataFrame, descriptor_names: List[str], scaler :StandardScaler=None) ->Tuple[pd.DataFrame, List[str]]:
    """規格化を行う。

    Args:
        df (pd.DataFrame): データ
        descriptor_names (List[str]): 説明変数名のリスト
        scaler (StandardScaler, optional): StandardScalerインスタンス. Defaults to None.

    Returns:
        pd.DataFrame: データ。
        [str]]: 規格化された説明変数名。
    """
    df = df.copy()
    
    # make labels of normalized values
    X_labels = []
    for i in descriptor_names:
        X_labels.append("s_{}".format(i))
    
    if X_labels[0] in df.columns:
        return df, X_labels, scaler
    
    #データ加工
    Xraw = df[descriptor_names].values
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(Xraw)
    X = scaler.transform(Xraw)    
    
    df_scaledX = pd.DataFrame(X,index=df.index, columns=X_labels)
    
    return pd.concat([df,df_scaledX], axis=1), X_labels, scaler

***次のデータで"s_"はいらない。***


In [ ]:
g_df_obs, g_X_labels, g_Xscaler = add_scaledX(g_df_obs, g_descriptor_names)
print("X_labels",g_X_labels)
g_df_new, g_X_labels, _ = add_scaledX(g_df_new, g_descriptor_names, scaler=g_Xscaler)
g_df_obs

In [ ]:
def add_pca(df, X_labels, ndim=2, drd=None, pca_prefix="pca"):
    """pca(ndim)を加える

    Args:
        df (pd.DataFrame)): データ。
        X_labels ([str]]): 説明変数名のリスト。
        ndim (int, optional): 圧縮後次元. Defaults to 2.
        drd (PCA), optional): PCAインスタンス. Defaults to None.
        pca_prefix (str, optional): 次元圧縮したカラム名. Defaults to "pca".

    Returns:
        [type]: [description]
    """
    df = df.copy()
    pca_labels = []
    for i in range(ndim):
        pca_labels.append("{}{}".format(pca_prefix, i+1))
    
    if pca_labels[0] in df.columns:
        return df, pca_labels, drd
    
    X = df[X_labels].values
    if drd is None:    
        drd = PCA(ndim)
        drd.fit(X)
    X2 = drd.transform(X)
    df_pca = pd.DataFrame(X2, index=df.index, columns=pca_labels)
    
    return pd.concat([df,df_pca], axis=1), pca_labels, drd

In [ ]:
# descriptor setを次元圧縮する。
if g_ndim_pre is not None:
    g_df_obs, g_pre_pca_labels, g_pre_pca = add_pca(g_df_obs, g_X_labels, ndim=g_ndim_pre, drd=None, 
                                              pca_prefix="pre_pca")
    
    g_df_new, _, _ = add_pca(g_df_new, g_X_labels, ndim=g_ndim_pre, drd=g_pre_pca, 
                                              pca_prefix="pre_pca")
    
    g_X_labels = g_pre_pca_labels
    print("new X_labels", g_X_labels)
    display(g_df_obs)

In [ ]:
g_df_obs.plot.scatter(x=g_X_labels[0],y=g_X_labels[1])

In [ ]:
def pca_contribution(df, X_labels):
    """寄与率の図示。

    Args:
        df (pd.DataFrame): データ。
        X_labels ([str]]): 説明変数のリスト。
    """
    X = df[X_labels].values
    
    # データ解析
    # 2次元へ次元圧縮
    explained_variance = []
    explained_variance_ratio = []
    ndim = X.shape[1]
    print("ndim",ndim)
    pca = PCA(ndim)
    pca.fit(X)
    print("explained_variance_ratio", pca.explained_variance_ratio_)
    
    indx = [i for i in range(1,len(pca.explained_variance_ratio_)+1)]
    esum = [np.sum(pca.explained_variance_ratio_[:i+1]) for i in 
            range(len(pca.explained_variance_ratio_))]
    
    fig, ax = plt.subplots()
    ax.plot(indx,pca.explained_variance_ratio_,"o-", label="explained_variance_ratio")
    ax.plot(indx,esum,"o-", label="sum(explained_variance_ratio)")
    ax.legend()
    
pca_contribution(g_df_obs, g_X_labels)

In [ ]:
g_df_obs[g_X_labels]

In [ ]:
def apply_clustering(df, X_labels,  action, nclusters=3, km=None, gmm=None):
    """クラスタリングを行う。

    Args:
        df (pd.DataFrame)): データ。
        X_labels ([str]]): 説明変数名のリスト。
        action ([str]]): kmeans, gmmのリスト。
        nclusters (int, optional): クラスター数. Defaults to 3.
        km (KMeans, optional): KMenas インスタンス. Defaults to None.
        gmm (GMM, optional): GMMインスタンス. Defaults to None.

    Returns:
        pd.DataFrame: データ。
        KMeans: KMeansインスタンス。
        GMM: GMMインスタンス。
    """
    df = df.copy()
    print("00")
    X = df[X_labels].values
    print("0")
    km = None
    gmm = None
    # データ解析, 3 class
    if "km" in action:
        if km is None:
            km = KMeans(nclusters)
            print("1")
            km.fit(X)
            print("2")
        yp_km = km.predict(X)
        print("3")
        df["km"] = yp_km
        print("4")
    if "gmm" in action:
        if gmm is None:
            gmm = GaussianMixture(nclusters)
            gmm.fit(X)
        yp_gmm = gmm.predict(X)
        yproba_gmm = gmm.predict_proba(X)
        df["gmm"] = yp_gmm
        df["yproba_gmm"] = yproba_gmm.tolist()
    return df, km, gmm


g_df_obs, g_km, g_gmm = apply_clustering(
    g_df_obs, g_X_labels, ACTCION, nclusters=g_nclusters)
print("-"*80)
g_df_new, g_km, g_gmm = apply_clustering(g_df_new, g_X_labels, ACTCION, 
                                         km=g_km, gmm=g_gmm)


**可視化**

まずデータを見ていく。
'a0.70_rp2.40', 'a0.70_rp3.00', 'a0.70_rp3.60', 'a0.70_rp4.20',
       'a0.70_rp4.80', 'a0.70_rp5.40'までが記述子であり、
最後の'key'は目的変数ではなく各行の説明をしているメタデータである。

In [ ]:
g_df_obs

可視化を含めて解析していく。
二次元で可視化するためにPCAにより次元圧縮を行っている。


In [ ]:
g_df_obs, g_pca_labels, g_pca = add_pca(g_df_obs, g_X_labels)
g_df_new, g_pca_labels, g_pca = add_pca(g_df_new, g_X_labels, drd=g_pca)

print("pca_labels", g_pca_labels)
g_df_obs

In [ ]:
def plot_X2(df: pd.DataFrame, columns: List[str], answer_label: str, 
            y_label: str, center : np.array, show_text=True, save_fig: bool=True,
           title: str="kmeans"):
    """二次元でのクラスタの表示。

    Args:
        df (pd.DataFrame): データ。
        columns (List[str]): x,y軸のカラム名。
        answer_label (str): 答えのカラム名。
        y_label (str): クラスタのカラム名。
        center (np.array): クラスター中心。
        show_text (bool, optional): 図で答えを表示するか。. Defaults to True.
        save_fig (bool, optional): 図を保存するか。Defaults to True
        title (str, optional): 図のtitleと図のファイル名の一部． Defaults to 'kmeans'.
    """
    X2 = df[columns].values
    marker = ["s", "o", "^"]
    color = ["green", "red", "blue"]
    xlim = X2[:, 0].min()-0.1, X2[:, 0].max()+0.1
    ylim = X2[:, 1].min()-0.1, X2[:, 1].max()+0.1
    fig, ax = plt.subplots()
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel(columns[0])
    ax.set_ylabel(columns[1])
    
    yp = df[y_label].values
    index = df[answer_label].values
    
    
    for xx, s, y in zip(X2, index, yp):
        ax.plot(xx[0], xx[1], marker[y], color=color[y])
        if show_text:
            ax.text(xx[0], xx[1], s)

    ax.scatter(center[:,0], center[:,1], s=100,  marker="x", color="black",)
            
    ax.set_title(y_label)
    fig.tight_layout()
    if save_fig:
        filename = f"image_executed/Fe2_{title}_clusters.png"
        fig.savefig(filename)
        print(f"saved to {filename}.")
    
for _this_action in ACTCION:
    if _this_action=="km":
        g_center = g_pca.transform(g_km.cluster_centers_)
    elif _this_action=="gmm":
        g_center = g_pca.transform(g_gmm.means_)
    plot_X2(g_df_obs, g_pca_labels,"polytype", _this_action, g_center, title=_this_action)

図中の■,▲,▼がクラスタリングした結果である。クラスタ中心をXで示す。

クラスタリング自体はこれでできています。

In [ ]:
from collections import Counter
from sklearn.metrics import confusion_matrix

def make_confusion_matrix(df: pd.DataFrame, answer_label: str, y_label: str):
    """yは文字、ypは数字なのでypの変換も行う。

    Args:
        df (pd.DataFrame): データ。
        answer_label (str): 答えのカラム名。
        y_label (str): 説明変数のカラム名。

    Raises:
        ValueError: 答えとクラスターのラベルの個数が合わない場合。

    Returns:
        pd.DataFrame: confusion matrix
    """
    conv_rule = {}
    y = df[answer_label].values
    uniq_y = np.unique(y)
    
    for y1 in uniq_y:
        dfq = df[df[answer_label]==y1]
        yp = dfq[y_label].values
        counter = Counter(yp)

        yp_mostcommon = counter.most_common()[0][0]
        conv_rule[yp_mostcommon] = y1
    print("yp most common", conv_rule)
    
    if len(conv_rule.keys())!=uniq_y.shape[0]:
        raise ValueError("len(conv_rule.keys())!=uniq_y.shape[0]")
        
    
    y = df[answer_label].values
    yp = df[y_label].values

    yp_conv = []
    for yp1 in yp:
        yp_conv.append(conv_rule[yp1])
    
    # We have y and yp_conv

    cm = confusion_matrix(y,yp_conv)
    index_list = []
    column_list = []
    for y1 in uniq_y:
        index_list.append("actual {}".format(y1))
        column_list.append("predict {}".format(y1))
    df_cm = pd.DataFrame(cm, index=index_list, columns=column_list)
    return df_cm
    

for _this_action in ACTCION:
    display(make_confusion_matrix(g_df_obs, "polytype", _this_action))


descriptor_set="a_rp", ndim_pre=Noneで
二次元図を見るとbccに思えるデータインスタンスが、hcpと「予測」されています。


### 問題

クラスタリングは次元圧縮後に行ったほうがうまく分割できることが多々あります。

- descriptor_setを変えてみる。
- pre_pca_labelsで次元圧縮する。


新規データに対しても同様にクラスタリングを適用してみる。

In [ ]:
for _this_action in ACTCION:
    if _this_action=="km":
        _center = g_pca.transform(g_km.cluster_centers_)
    elif _this_action=="gmm":
        _center = g_pca.transform(g_gmm.means_)
    plot_X2(g_df_new, g_pca_labels, "polytype", _this_action, _center, title="new_data_"+_this_action)


gaussian mixture modelを用いると予測値が確率でも出力される。
●の濃淡でその確率を示す。

In [ ]:
import seaborn as sns

def plot_contour(df, pca_labels, yproba_label):
    """2Dで各クラスターの確率表示を行う。

    Args:
        df (pd.DataFrame):　データ。
        pca_labels ([str]): 次元圧縮を行った説明変数カラム名。
        yproba_label ([tr): 確率のカラム名
    """
    X2 = df[pca_labels].values
    yproba = df[yproba_label].values
    
    # change type to np.array
    yproba2 = []
    for yp in yproba:
        yproba2.append(yp)
    yproba = np.array(yproba2)
    
    plt.figure()
    for i in range(3):
        colors = ["red", "blue", "green"]
        fig, ax = plt.subplots()
        cmap = sns.light_palette(colors[i], as_cmap=True)
        plot = plt.scatter(X2[:, 0], X2[:, 1],
                           c=yproba[:, i], cmap=cmap, label=str(i))
        fig.legend()
        fig.colorbar(plot)
        fig.show()

if "gmm" in ACTCION:
    plot_contour(g_df_obs, g_pca_labels, "yproba_gmm")

### 補足

#### シルエット図
kmeanクラスタリングでクラスタ数などのパラメータを視覚的に評価する方法。

横軸：１に近いほど分離が良い。
縦軸：クラスター毎のシルエットプロット。

見方：シルエットプロットの太さが同じ（縦方向）の厚さに成るほどよい。


scikit-learnのシルエット図を書くscriptを用いてシルエット図を書く。

https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html#example-cluster-plot-kmeans-silhouette-analysis-py%5D

In [ ]:
from sklearn.metrics import   silhouette_samples, silhouette_score
import matplotlib.cm as cm
import numpy as np

def make_silhouette_plot( X, range_n_clusters = [2, 3, 4, 5, 6]):
    """from sklearn
    """
    for n_clusters in range_n_clusters:
        fig, (ax1, ax2) = plt.subplots(1, 2)
        fig.set_size_inches(18, 7)
        ax1.set_xlim([-0.1, 1])
        ax1.set_ylim([0, len(X) + (n_clusters + 1) * 10])

        clusterer = KMeans(n_clusters=n_clusters, random_state=10)
        cluster_labels = clusterer.fit_predict(X)
        silhouette_avg = silhouette_score(X, cluster_labels)
        print(
            "For n_clusters =",
            n_clusters,
            "The average silhouette_score is :",
            silhouette_avg,
        )
        sample_silhouette_values = silhouette_samples(X, cluster_labels)
        y_lower = 10
        for i in range(n_clusters):
            # Aggregate the silhouette scores for samples belonging to
            # cluster i, and sort them
            ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]

            ith_cluster_silhouette_values.sort()

            size_cluster_i = ith_cluster_silhouette_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = cm.nipy_spectral(float(i) / n_clusters)
            ax1.fill_betweenx(
                np.arange(y_lower, y_upper),
                0,
                ith_cluster_silhouette_values,
                facecolor=color,
                edgecolor=color,
                alpha=0.7,
            )
            ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
            y_lower = y_upper + 10  # 10 for the 0 samples
        ax1.set_title("The silhouette plot for the various clusters.")
        ax1.set_xlabel("The silhouette coefficient values")
        ax1.set_ylabel("Cluster label")

        # The vertical line for average silhouette score of all the values
        ax1.axvline(x=silhouette_avg, color="red", linestyle="--")

        ax1.set_yticks([])  # Clear the yaxis labels / ticks
        ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

        # 2nd Plot showing the actual clusters formed
        colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
        ax2.scatter(
            X[:, 0], X[:, 1], marker=".", s=30, lw=5, alpha=0.7, c=colors, edgecolor="k"
        )

        # Labeling the clusters
        centers = clusterer.cluster_centers_
        # Draw white circles at cluster centers
        ax2.scatter(
            centers[:, 0],
            centers[:, 1],
            marker="o",
            c="white",
            alpha=1,
            s=200,
            edgecolor="k",
        )

        for i, c in enumerate(centers):
            ax2.scatter(c[0], c[1], marker="$%d$" % i, alpha=1, s=50, edgecolor="k")

        ax2.set_title("The visualization of the clustered data.")
        ax2.set_xlabel("Feature space for the 1st feature")
        ax2.set_ylabel("Feature space for the 2nd feature")

        plt.suptitle(
            "Silhouette analysis for KMeans clustering on sample data with n_clusters = %d"
            % n_clusters,
            fontsize=14,
            fontweight="bold",
        )

g_X = g_df_obs[g_X_labels].values
make_silhouette_plot( g_X,)

上によるとクラスター数３が最も良い。

（ただし、厚さが同じなのは各クラスターすうを同じ数にしたことが原因。）

#### elbow法


In [ ]:
"""from stack overflow
https://stackoverflow.com/questions/41540751/sklearn-kmeans-equivalent-of-elbow-method
"""
def elblow_method(X, ncluster_max = 10, random_state=1):
    """eblow法

    Args:
        X (np.ndarray)): 説明変数
        ncluster_max (int, optional): 最大クラスタ多数. Defaults to 10.
        random_state (int, optional): KMeansのrandom_state. Defaults to 1.
    """
    result = []

    ncluster_list = range(1,ncluster_max)
    for i  in ncluster_list:               # 1~ncluster_maxクラスタまで一計算 
        km = KMeans(n_clusters=i,
                    random_state=random_state)
        km.fit(X)                         # クラスタリングの計算を実行
        result.append(km.inertia_)   # km.fitするとkm.inertia_が得られる
        # result.append(km.score(X))
        
    plt.plot(ncluster_list, result,marker='o')
    plt.xlabel('nclusters')
    plt.ylabel('Distortion')
    plt.title('Elbow curve')

    plt.show()
    
g_X = g_df_obs[g_X_labels].values
elblow_method(g_X)

さちるnlusterを適当とする。